# BRL Pre-CDI Swap: Market Data, Conventions & Pricing

This notebook walks through:
1. **Where** BRL market data comes from (BCB, B3, ANBIMA)
2. **How** BRL rate conventions work (BUS/252, CDI, DI futures)
3. **Transforming** observable rates into discount factors
4. **Building** a `MarketData` snapshot from real April 2026 data
5. **Pricing** a swap and **shocking** the curve

---

## 1. Data Sources

| Source | What it provides | Access |
|--------|-----------------|--------|
| **BCB** (Banco Central do Brasil) | CDI overnight rate, Selic target, PTAX (USD/BRL) | Free API: `api.bcb.gov.br/dados/serie/bcdata.sgs.{id}/dados` |
| **B3** (Brasil Bolsa Balcão) | DI futures settlement prices → term structure | B3 Market Data Portal (registration required) |
| **ANBIMA** | Official yield curves (PRE, IPCA), holiday calendar | Members-only; curves published daily |
| **Trading Economics** | Government bond yields (proxy for DI curve) | `tradingeconomics.com/brazil/government-bond-yield` |

### BCB API Examples

```
CDI daily rate:     series 12    → 0.054266% (Apr 10, 2026)
Selic target rate:  series 432   → 14.75%
USD/BRL PTAX:       series 1     → 5.0229
```

## 2. BRL Rate Conventions

### The BUS/252 Day Count

Unlike most global rates (ACT/360, ACT/365), Brazil uses **business days / 252**:

$$\alpha = \frac{\text{business days}}{252}$$

Why 252? Brazil has ~252 business days per year (365 minus weekends minus ~13 ANBIMA holidays). This convention is universal across BRL fixed income:
- DI futures
- Pre-CDI swaps
- NTN-B (inflation-linked bonds)
- Corporate debentures

**Implication:** You need a Brazilian holiday calendar to count business days exactly. Without one, approximate: `bd ≈ calendar_days × 252/365.25`.

### CDI (Certificado de Depósito Interbancário)

The CDI rate is Brazil's overnight interbank rate — analogous to SOFR or Fed Funds.

- Published daily by B3
- Quoted as a **daily percentage**, e.g., 0.054266%
- Annualized: $(1 + 0.00054266)^{252} - 1 \approx 14.65\%$
- Closely tracks the Selic target (currently 14.75%)

### DI Futures (DI1)

The DI futures contract on B3 is the primary instrument for BRL rate term structure.

- **Notional:** 100,000 PU (preço unitário) at maturity
- **Price:** $PU = \frac{100{,}000}{(1 + r)^{bd/252}}$ where $r$ is the DI rate, $bd$ = business days to maturity
- **Discount factor:** $df(t) = \frac{1}{(1+r)^{bd/252}}$

This is the formula we use to convert observable DI rates into discount factors.

## 3. From Observable Rates to Discount Factors

Let's walk through the math step by step.

In [1]:
# Real market data as of April 10, 2026
# Source: Brazil government bond yields (tradingeconomics.com)
# These closely track DI futures settlement rates

brl_rates = {
    "3M":  0.1404,   # 14.04%
    "6M":  0.1408,   # 14.08%
    "1Y":  0.1370,   # 13.70%
    "2Y":  0.1357,   # 13.57%
    "3Y":  0.1348,   # 13.48%
    "5Y":  0.1354,   # 13.54%
    "10Y": 0.1372,   # 13.72%
}

print("BRL DI Curve (Apr 10, 2026)")
print("============================")
print(f"{'Tenor':<6} {'Rate':>8} {'Approx BD':>10} {'DF':>12}")
print("-" * 40)

for tenor, rate in brl_rates.items():
    # Approximate business days
    if tenor.endswith("M"):
        months = int(tenor[:-1])
        cal_days = round(months * 30.4375)
    else:
        years = int(tenor[:-1])
        cal_days = round(years * 365.25)
    
    bus_days = round(cal_days * 252 / 365.25)
    df = 1.0 / (1.0 + rate) ** (bus_days / 252)
    
    print(f"{tenor:<6} {rate:>7.2%} {bus_days:>10} {df:>12.8f}")

BRL DI Curve (Apr 10, 2026)
Tenor      Rate  Approx BD           DF
----------------------------------------
3M      14.04%         63   0.96768876
6M      14.08%        126   0.93625736
1Y      13.70%        252   0.87950748
2Y      13.57%        504   0.77530529
3Y      13.48%        756   0.68429288
5Y      13.54%       1260   0.52997520
10Y     13.72%       2520   0.27645948


### Key Observations

1. **The curve is inverted at the front.** 3M (14.04%) and 6M (14.08%) are higher than 1Y (13.70%). This reflects the market pricing Selic cuts — the central bank is at 14.75% but expected to ease.

2. **Minimum around 3Y (13.48%).** The belly of the curve is the trough, with rates rising again at 10Y (13.72%) — a classic post-tightening shape.

3. **Business days matter.** A 1-year tenor has ~252 business days, not 365. Using calendar days would give the wrong discount factor by ~3%.

## 4. Building the MarketData Snapshot

Our library separates DATA from ACTIONS. The `bootstrap` module (CALCULATION) converts rates to DFs. The `snapshots` module (ACTION) encodes today's real rates.

In [1]:
from fi_claude.market.snapshots import april_10_2026_snapshot

market = april_10_2026_snapshot()

print(f"Valuation date: {market.valuation_date}")
print(f"Discount curves: {list(market.discount_curves.keys())}")
print(f"Inflation curves: {list(market.inflation_curves.keys())}")
print(f"FX spots: {market.fx_spot_rates}")
print()

# Inspect the BRL-CDI curve
brl_curve = market.discount_curves["BRL-CDI"]
print(f"BRL-CDI curve: {len(brl_curve.nodes)} nodes")
print(f"{'Date':<14} {'DF':>12} {'Impl Rate':>10}")
print("-" * 38)
for node in brl_curve.nodes:
    days = (node.date - brl_curve.reference_date).days
    bd = round(days * 252 / 365.25)
    if bd > 0:
        implied_rate = (1.0 / node.value) ** (252.0 / bd) - 1.0
    else:
        implied_rate = 0.0
    print(f"{node.date}   {node.value:>12.8f} {implied_rate:>9.2%}")

Valuation date: 2026-04-10
Discount curves: ['BRL-CDI', 'USD', 'EUR', 'MXN-TIIE']
Inflation curves: ['US-CPI']
FX spots: {'USD/BRL': 5.0229, 'EUR/USD': 1.095, 'USD/MXN': 18.8}

BRL-CDI curve: 8 nodes
Date                     DF  Impl Rate
--------------------------------------
2026-04-10     1.00000000     0.00%
2026-07-10     0.96768876    14.04%
2026-10-10     0.93625736    14.08%
2027-04-10     0.87950748    13.70%
2028-04-09     0.77530529    13.57%
2029-04-10     0.68429288    13.48%
2031-04-10     0.52997520    13.54%
2036-04-09     0.27645948    13.72%


### What's in the snapshot?

| Curve | Source | Convention |
|-------|--------|------------|
| `BRL-CDI` | DI futures / govt bonds | Annual compounding, BUS/252 |
| `USD` | US Treasuries | Semi-annual compounding, ACT/365 |
| `EUR` | German Bunds | Semi-annual compounding, ACT/365 |
| `MXN-TIIE` | TIIE 28-day + term structure | Annual compounding, ACT/360 |
| `US-CPI` | BLS CPI-U | Index levels, linearly interpolated |
| FX spots | BCB PTAX, market | USD/BRL=5.0229, EUR/USD=1.095 |

## 5. Pricing a BRL Pre-CDI Swap

A Pre-CDI swap exchanges a single payment at maturity (OpenGamma QR n.18, Arroub 2013):
- **Fixed leg payoff at maturity:** $N \times (1 + k)^{n\delta}$ where $\delta = 1/252$, $n$ = business days
- **Float leg payoff at maturity:** $N \times \prod_{i=0}^{n-1} (1 + DI_i)^{\delta}$ (daily CDI compounding)

### Present Value (single-curve, OpenGamma QR n.18 §4-5)

In a single-curve environment (discount curve = CDI curve), the overnight compounded product **telescopes**:

$$PV_{\text{float}} = N \times P^D(t, t_0)$$

The float leg PV equals notional times the discount factor to the start date. For a spot-starting swap, $P^D(t, t_0) = 1$, so **the float leg is simply worth par**.

The fixed leg PV is the maturity payoff discounted back:

$$PV_{\text{fixed}} = N \times (1+k)^{n\delta} \times P^D(t, t_n)$$

### NPV

$$NPV_{\text{pay fixed}} = PV_{\text{float}} - PV_{\text{fixed}} = N \times P^D(t, t_0) - N \times (1+k)^{\alpha} \times P^D(t, t_n)$$

### Par Rate

At inception, NPV = 0, so: $(1+k_{\text{par}})^{\alpha} = \frac{P^D(t, t_0)}{P^D(t, t_n)}$

$$k_{\text{par}} = \left(\frac{P^D(t, t_0)}{P^D(t, t_n)}\right)^{1/\alpha} - 1$$

For spot-starting swaps ($P^D(t, t_0) = 1$): $k_{\text{par}} = P^D(t, t_n)^{-1/\alpha} - 1$

In [2]:
from datetime import date
from decimal import Decimal

from fi_claude.data.instruments import BrlPreCdiSwap
from fi_claude.data.common import PayReceive
from fi_claude.pricers.brl_pre_cdi import price_brl_pre_cdi_swap

# A 2-year BRL Pre-CDI swap
# Paying fixed at 13.50% (below the 2Y market rate of 13.57%)
# → we expect a small positive NPV (we locked in a rate below market)
swap = BrlPreCdiSwap(
    notional=Decimal("10_000_000"),   # BRL 10 million
    fixed_rate=0.1350,                # 13.50% — below market
    start_date=date(2026, 4, 10),
    end_date=date(2028, 4, 10),
    pay_receive_fixed=PayReceive.PAY,  # pay fixed, receive CDI
)

# Business days for a 2-year period
# 730 cal days × 252/365.25 ≈ 504 business days
business_days = 504

result = price_brl_pre_cdi_swap(swap, market, business_days)

print(f"Instrument:  BRL Pre-CDI Swap")
print(f"Direction:   Pay {swap.fixed_rate:.2%} fixed, receive CDI")
print(f"Notional:    BRL {swap.notional:,.0f}")
print(f"Period:      {swap.start_date} → {swap.end_date}")
print(f"Bus days:    {business_days}")
print()
print(f"NPV:         BRL {result.present_value:,.2f}")
print()
print("Details:")
for k, v in result.details.items():
    print(f"  {k:>15}: {v}")

Instrument:  BRL Pre-CDI Swap
Direction:   Pay 13.50% fixed, receive CDI
Notional:    BRL 10,000,000
Period:      2026-04-10 → 2028-04-10
Bus days:    504

NPV:         BRL 2,918,276.59

Details:
     fixed_leg_pv: 9984269.61
     float_leg_pv: 12902546.2
    year_fraction: 2.0
         df_start: 1.0
           df_end: 0.77504082


### Intuition Check

We're paying fixed at 13.50% while the market 2Y rate is 13.57%. We pay *below* the market rate, so the swap has positive NPV to us.

With the corrected formula (float PV = N * df_start = N for spot-starting), the NPV reflects only the rate differential, not a compounding artifact.

## 6. Shocking the Curve

Now let's use the shock suite to see how the swap responds to various curve movements.

### Shock mechanics

The fundamental discount factor bump:

$$df'(t) = df(t) \times e^{-\Delta r \cdot t / 10000}$$

where $\Delta r$ is the shock in basis points. This is exact for continuously-compounded rates and a close approximation for annually-compounded rates over typical shock sizes.

In [3]:
from fi_claude.risk.shocks import (
    apply_shocks, parallel, steepen, point_shock, scenario
)

def reprice(shocked_market, label):
    """Reprice the swap under a shocked market."""
    res = price_brl_pre_cdi_swap(swap, shocked_market, business_days)
    delta = float(res.present_value - result.present_value)
    print(f"{label:<40} NPV = BRL {res.present_value:>12,.2f}  (Δ = {delta:>+10,.2f})")
    return res

print(f"{'Scenario':<40} {'NPV':>20}  {'Change':>12}")
print("=" * 78)

# Base case
reprice(market, "Base")

# 1. Parallel shift +25bps → rates up → fixed leg worth less → NPV improves for pay-fixed
s1 = scenario("CDI +25bp", rates=parallel("BRL-CDI", 25))
reprice(apply_shocks(market, s1), "CDI parallel +25bp")

# 2. Parallel shift -50bps → rates down → NPV deteriorates for pay-fixed
s2 = scenario("CDI -50bp", rates=parallel("BRL-CDI", -50))
reprice(apply_shocks(market, s2), "CDI parallel -50bp")

# 3. Curve steepening: short end -25bp, long end +25bp
s3 = scenario("Steepen", rates=steepen("BRL-CDI", short_bps=-25, long_bps=25))
reprice(apply_shocks(market, s3), "CDI steepen (-25/+25)")

# 4. Point shock at 2Y (+50bp) — right where our swap matures
s4 = scenario("2Y bump", rates=point_shock("BRL-CDI", target_years=2.0, bps=50))
reprice(apply_shocks(market, s4), "CDI 2Y point +50bp")

# 5. Large parallel shock +100bp (stress test)
s5 = scenario("Stress +100", rates=parallel("BRL-CDI", 100))
reprice(apply_shocks(market, s5), "CDI parallel +100bp (stress)")

Scenario                                                  NPV        Change
Base                                     NPV = BRL 2,918,276.59  (Δ =      +0.00)
CDI parallel +25bp                       NPV = BRL 3,032,826.00  (Δ = +114,549.41)
CDI parallel -50bp                       NPV = BRL 2,689,394.07  (Δ = -228,882.52)
CDI steepen (-25/+25)                    NPV = BRL 2,817,285.40  (Δ = -100,991.19)
CDI 2Y point +50bp                       NPV = BRL 3,146,637.83  (Δ = +228,361.24)
CDI parallel +100bp (stress)             NPV = BRL 3,376,941.26  (Δ = +458,664.67)


PricingResult(instrument_type='BRL_PRE_CDI_SWAP', valuation_date=datetime.date(2026, 4, 10), currency=<Currency.BRL: 'BRL'>, present_value=Decimal('3376941.26'), cashflows=(), details={'fixed_leg_pv': 9786433.86, 'float_leg_pv': 13163375.12, 'year_fraction': 2.0, 'df_start': 1.0, 'df_end': 0.75968358})

### Reading the Results

**Pay-fixed swap sensitivity:**
- Rates **up** → NPV **increases** (we locked in a lower rate; market moved in our favor)
- Rates **down** → NPV **decreases** (our locked rate is now above market)
- **Point shock at 2Y** has the most impact because that's where our swap matures
- **Steepener** has mixed effects: the short end helps (lower start DF) but the long end hurts less (our maturity is only 2Y)

## 7. DV01 — Rate Sensitivity

DV01 = change in NPV for a 1bp parallel shift. This is the swap's rate sensitivity.

In [4]:
# DV01: NPV change per 1bp parallel shift
up_1bp = apply_shocks(market, scenario(rates=parallel("BRL-CDI", 1)))
dn_1bp = apply_shocks(market, scenario(rates=parallel("BRL-CDI", -1)))

npv_up = float(price_brl_pre_cdi_swap(swap, up_1bp, business_days).present_value)
npv_dn = float(price_brl_pre_cdi_swap(swap, dn_1bp, business_days).present_value)

dv01 = (npv_up - npv_dn) / 2.0  # central difference

print(f"DV01 = BRL {dv01:,.2f} per 1bp parallel shift")
print(f"")
print(f"Interpretation: for every 1bp increase in the CDI curve,")
print(f"the pay-fixed swap gains approximately BRL {dv01:,.2f}")
print(f"")
print(f"Notional:  BRL {float(swap.notional):>14,.0f}")
print(f"DV01/Notl: {dv01 / float(swap.notional) * 10000:.4f} bp")

DV01 = BRL 4,580.50 per 1bp parallel shift

Interpretation: for every 1bp increase in the CDI curve,
the pay-fixed swap gains approximately BRL 4,580.50

Notional:  BRL     10,000,000
DV01/Notl: 4.5805 bp


## 8. Seasoning — Theta, Carry, Rolldown

How does the swap's value evolve as time passes?

In [7]:
from functools import partial
from fi_claude.risk.seasoning import compute_seasoning

# Bind the swap and business_days to the pricer for the seasoning API
pricer = partial(price_brl_pre_cdi_swap, swap, business_days=business_days)

# 1-week and 1-month horizon
for days in [7, 30]:
    h = compute_seasoning(
        pricer, market, horizon_days=days,
        rate_curve_keys=("BRL-CDI",),  # needed for DV01 calculation
    )
    base_dv01 = h.base_risk.dv01.get("BRL-CDI", 0.0)
    horizon_dv01 = h.horizon_risk.dv01.get("BRL-CDI", 0.0)
    print(f"--- {days}-day horizon ---")
    print(f"  Base NPV:     BRL {h.base_pv:>12,.2f}")
    print(f"  Horizon NPV:  BRL {h.horizon_pv:>12,.2f}")
    print(f"  Total theta:  BRL {h.total_theta:>12,.2f}")
    print(f"  Carry:        BRL {h.carry:>12,.2f}")
    print(f"  Rolldown:     BRL {h.rolldown:>12,.2f}")
    print(f"  Base DV01:    BRL {base_dv01:>12,.2f}")
    print(f"  Horizon DV01: BRL {horizon_dv01:>12,.2f}")
    print()

--- 7-day horizon ---
  Base NPV:     BRL 2,918,276.59
  Horizon NPV:  BRL 2,863,033.08
  Total theta:  BRL   -55,243.51
  Carry:        BRL         0.00
  Rolldown:     BRL   -55,243.51
  Base DV01:    BRL     4,580.56
  Horizon DV01: BRL     4,535.31

--- 30-day horizon ---
  Base NPV:     BRL 2,918,276.59
  Horizon NPV:  BRL 2,681,310.27
  Total theta:  BRL  -236,966.32
  Carry:        BRL         0.00
  Rolldown:     BRL  -236,966.32
  Base DV01:    BRL     4,580.56
  Horizon DV01: BRL     4,386.99



## 9. Convention Comparison: BRL vs USD vs EUR

The same 2-year maturity produces very different discount factors under different conventions:

| Currency | Convention | Compounding | Rate | df(2Y) |
|----------|-----------|-------------|------|--------|
| BRL | BUS/252 | Annual | 13.57% | $(1+0.1357)^{-504/252}$ |
| USD | ACT/365 | Semi-annual | 3.80% | $(1+0.0380/2)^{-4}$ |
| EUR | ACT/365 | Semi-annual | 2.58% | $(1+0.0258/2)^{-4}$ |

In [8]:
from fi_claude.curves.bootstrap import (
    discount_factor_from_di_rate,
    discount_factor_from_yield,
)

# 2-year discount factors under each convention
df_brl = discount_factor_from_di_rate(0.1357, 504)        # BUS/252, annual
df_usd = discount_factor_from_yield(0.03801, 2.0, "semi-annual")  # Treasury
df_eur = discount_factor_from_yield(0.02584, 2.0, "semi-annual")  # Bund

print(f"2-Year Discount Factors")
print(f"=======================")
print(f"BRL (BUS/252, 13.57%):     {df_brl:.8f}")
print(f"USD (semi-ann, 3.80%):     {df_usd:.8f}")
print(f"EUR (semi-ann, 2.58%):     {df_eur:.8f}")
print()
print(f"BRL discounts 2Y cashflows by {(1-df_brl)*100:.1f}%")
print(f"USD discounts 2Y cashflows by {(1-df_usd)*100:.1f}%")
print(f"EUR discounts 2Y cashflows by {(1-df_eur)*100:.1f}%")
print()
print("→ A BRL 2Y cashflow is worth ~23% less in PV terms.")
print("  A EUR 2Y cashflow is worth ~5% less. Night and day.")

2-Year Discount Factors
BRL (BUS/252, 13.57%):     0.77530529
USD (semi-ann, 3.80%):     0.92745904
EUR (semi-ann, 2.58%):     0.94994709

BRL discounts 2Y cashflows by 22.5%
USD discounts 2Y cashflows by 7.3%
EUR discounts 2Y cashflows by 5.0%

→ A BRL 2Y cashflow is worth ~23% less in PV terms.
  A EUR 2Y cashflow is worth ~5% less. Night and day.


## 10. At-Market vs Off-Market Swaps

An **at-market** swap has NPV ≈ 0 at inception (the fixed rate equals the market rate). Let's find it.

In [ ]:
from fi_claude.curves.interpolation import interpolate_discount_factor

# Closed-form par rate: k_par = (df_start / df_end)^(1/alpha) - 1
df_start = interpolate_discount_factor(brl_curve, swap.start_date)
df_end = interpolate_discount_factor(brl_curve, swap.end_date)
alpha = business_days / 252

par_rate = (df_start / df_end) ** (1.0 / alpha) - 1.0

print(f"Par swap rate (2Y, closed-form)")
print(f"================================")
print(f"  df(start) = {df_start:.8f}")
print(f"  df(end)   = {df_end:.8f}")
print(f"  alpha     = {alpha:.4f}")
print(f"  k_par     = (df_start/df_end)^(1/alpha) - 1")
print(f"            = ({df_start:.8f} / {df_end:.8f})^(1/{alpha}) - 1")
print(f"            = {par_rate:.4%}")
print()

# Verify: price at par rate should give NPV ≈ 0
par_swap = swap.model_copy(update={"fixed_rate": par_rate})
par_result = price_brl_pre_cdi_swap(par_swap, market, business_days)
print(f"Verification: NPV at par rate = BRL {par_result.present_value:,.2f}")
print(f"Market 2Y rate:                 13.57%")
print(f"Computed par rate:              {par_rate:.2%}")
print()
print("The small residual comes from log-linear interpolation")
print("between the 1Y and 3Y curve nodes (the 2Y swap end date")
print("falls between them).")

## Summary

### The Pipeline: Raw Data → Pricing

```
BCB API / B3         bootstrap.py          market.py          pricers/
───────────     ──────────────────     ──────────────     ──────────────
DI rate: 13.57%  → df = 1/(1+r)^(bd/252) → DiscountCurve → price_brl_pre_cdi()
CDI: 0.054%/day     CALCULATION              DATA            CALCULATION
PTAX: 5.0229                                                    ↓
  ACTION                                                  PricingResult
```

### Pricing Formula (OpenGamma QR n.18)

$$NPV_{\text{pay fixed}} = N \times P^D(t, t_0) \;-\; N \times (1+k)^{\alpha} \times P^D(t, t_n)$$

The key insight: the overnight compounded float leg **telescopes** to $N \times P^D(t, t_0)$ under single-curve assumptions (discount = forward). This is proven by iterated application of the tower property under the $t_n$-forward measure (QR n.18 §4).

### BRL-Specific Gotchas

1. **BUS/252 everywhere.** Never use ACT/365 for BRL rates.
2. **Holiday calendar is essential.** ANBIMA publishes it annually. Without it, approximate with the 252/365.25 ratio.
3. **DI rates are zero-coupon.** Each DI futures maturity gives a direct zero rate — no bootstrapping needed (unlike USD swap curves where you must strip coupon bonds).
4. **CDI ≠ Selic.** CDI tracks Selic closely but can diverge by a few bp during liquidity stress.
5. **Annual compounding.** BRL uses $(1+r)^t$, not $e^{rt}$ or $(1+r/2)^{2t}$. The bump formula $df' = df \cdot e^{-\Delta r \cdot t}$ is an approximation that's accurate for small shocks.

### References

- **OpenGamma QR n.18** — "Brazilian Swaps" (Arroub, Dec 2013): https://quant.opengamma.io/Brazilian-Swaps-OpenGamma.pdf
- **OpenGamma QR** — "Interest Rate Instruments and Market Conventions" (Henrard, Dec 2013): https://quant.opengamma.io/Interest-Rate-Instruments-and-Market-Conventions.pdf
- **Strata** — `DiscountingSwapTradePricer`: https://strata.opengamma.io/swap/